# Proceso del dato: del archivo descargado a lo que entra en la red

**Finalidad**: ver, etapa a etapa, qué le pasa al dato antes de llegar a la red.
**Flujo**: Archivo → Huecos → Relleno → Recorte → Transformación → Escalado → Ventana
**Requiere**: `make data` y `make prepare`. No entrena ni escribe nada.
**Decisiones que ilustra**: apartados 1 a 4 de `docs/DECISIONES.md`.

El escalado se ajusta solo con train. Las figuras de limpieza muestran la serie completa: describen el archivo, no eligen nada.

In [ ]:
import json

import pandas as pd

from oil_rnn import explore as ex
from oil_rnn import explore_plots as plots
from oil_rnn.config import load_config
from oil_rnn.paths import SPLITS_MANIFEST

cfg = load_config()
time_col, target = cfg["data"]["time_column"], cfg["data"]["target"]
manifest = json.loads(SPLITS_MANIFEST.read_text(encoding="utf-8"))

stages = ex.pipeline_stages(cfg)
resumen = pd.DataFrame(
    {
        "filas": {k: len(v) for k, v in stages.items()},
        "celdas vacías": {k: int(v.isna().sum().sum()) for k, v in stages.items()},
        "primera fecha": {k: v[time_col].min().date() for k, v in stages.items()},
        "última fecha": {k: v[time_col].max().date() for k, v in stages.items()},
    }
)
resumen.index = [ex.STAGE_LABELS[k] for k in resumen.index]
resumen

## 1. Archivo: qué columnas tienen huecos y cuándo

Cada celda es el porcentaje de filas sin dato de una columna en un año. Hay dos tipos de hueco:

- **Al principio** (bandas oscuras a la izquierda): la serie aún no existía. El futuro del Brent empieza en 2007.
- **Interiores** (tonos claros): festivos distintos en cada mercado y días sin cotización.

Las filas son solo los días con Brent (`data.source.calendar`), por eso `brent` no tiene huecos.

In [ ]:
plots.missing_map(ex.missing_by_year(stages["bruto"], time_col))

In [ ]:
raw = stages["bruto"].set_index(time_col)
habiles = pd.bdate_range(raw.index.min(), raw.index.max())
sin_brent = len(habiles) - len(raw)
print(f"Días hábiles: {len(habiles)} · con Brent: {len(raw)} · sin Brent: {sin_brent}")

primer = raw.apply(lambda s: s.first_valid_index())
interiores = {c: int(raw.loc[primer[c] :, c].isna().sum()) for c in raw.columns}
pd.DataFrame({"primer dato": primer.dt.date, "huecos interiores": interiores}).sort_values(
    "huecos interiores", ascending=False
)

**Qué sostiene**: decisiones 1.2 (calendario), 1.4 (futuro del Brent excluido) y 2.2 (los datos empiezan en la primera fila completa).

## 2. Relleno: qué hace `ffill` con un hueco interior

El punto naranja es un día sin dato al que se le copia el valor anterior. Al pasar a rendimientos, ese día vale 0: la red no distingue «no hubo dato» de «no cambió». Cambia `columna` y las fechas para ver otros casos.

In [ ]:
columna, desde, hasta = "henry_hub", "2019-11-01", "2020-02-01"

antes, despues = stages["corte"], stages["relleno"]
tramo = (antes[time_col] >= desde) & (antes[time_col] <= hasta)
plots.fill_effect(antes[tramo], despues[tramo], time_col, columna)

**Qué sostiene**: decisión 2.1 (`ffill`, valorada Regular) y la mejora «indicador de disponibilidad» del apartado 12.

## 3. Transformación y escalado: una variable en sus cuatro formas

1. **Valor publicado**: no es estacionario; su nivel cambia de época en época.
2. **Transformada**: cambio diario, `log(x[t]/x[t−1])`. Media y dispersión estables.
3. **Estandarizada**: se resta la media y se divide por la desviación, las dos de train.
4. **Recortada**: limitada a ±5 desviaciones (líneas naranjas). Es lo que entra en la red.

El `Preprocessor` es el mismo objeto que usa el entrenamiento.

In [ ]:
from oil_rnn.data import load_split, load_train_val
from oil_rnn.features import Preprocessor

train = load_split("train")
prep = Preprocessor(cfg).fit(train)  # media y desviación: solo train
print(f"{prep.n_features} entradas: {prep.inputs}")

dev = load_train_val()  # train + val: solo para dibujar; test no aparece
formas = ex.input_stages(prep, dev)
plots.input_forms(formas, "brent", prep.clip)

El WTI cotizó a −37,63 $ el 20/04/2020. Antes de transformar se le pone un suelo de 1 $ (`data.inputs.floor`); aun así el rendimiento de ese día es enorme y el recorte lo limita.

In [ ]:
plots.input_forms(formas, "wti", prep.clip)

In [ ]:
z = formas["estandarizada"]
pd.DataFrame(
    {
        "máximo en desviaciones": z.abs().max().round(1),
        "días recortados": (z.abs() > prep.clip).sum(),
        "transformación": dict(zip(prep.features, prep.transforms, strict=True)),
    }
).sort_values("máximo en desviaciones", ascending=False)

**Qué sostiene**: decisiones 2.5 (suelo), 3.1 (rendimientos), 3.3 (escalado con train) y 3.4 (recorte).

## 4. Ventana: qué ve la red un día concreto

Cada predicción usa las 60 filas que terminan el día del pronóstico. La red recibe esta matriz y nada más: no ve el nivel del precio ni la fecha. Cambia `fecha` (cualquier día de train o val).

In [ ]:
fecha = "2008-07-03"  # máximo de 2008

win = ex.window_at(prep, dev, fecha)
print(
    f"Día del pronóstico: {win['origin']:%d/%m/%Y} · último precio conocido: {win['y_last']:.2f} $"
)
print(f"Día que se predice: {win['target']:%d/%m/%Y} ({prep.horizon} filas después)")
print(f"Precio real ese día: {win['y_true']:.2f} $")
print(f"La red tiene que acertar: log({win['y_true']:.2f} / {win['y_last']:.2f}) = {win['y']:+.4f}")
plots.window_heatmap(win)

In [ ]:
plots.window_on_price(dev, time_col, target, win)

La línea discontinua gris es la persistencia: «dentro de h días, el precio de hoy». La red predice cuánto se aparta el precio de esa línea, en proporción. Una salida de 0 es exactamente la persistencia.

Prueba el horizonte largo:

In [ ]:
prep_252 = Preprocessor(load_config(overrides=["data.horizon=252"])).fit(train)
plots.window_on_price(dev, time_col, target, ex.window_at(prep_252, dev, fecha), margin=200)

**Qué sostiene**: decisiones 3.5 (lo que aprende la red), 4.1 (ventana de 60) y 4.2 (alineación).

## 5. Resumen: lo que hay que saber decir

- El dato pasa por 4 etapas de limpieza y 3 de transformación; ninguna usa valores posteriores.
- Lo único que se ajusta con datos es la media y la desviación del escalado, con train.
- Dos puntos débiles: `ffill` convierte huecos en rendimientos 0, y la última fila tiene 8 días de retraso (decisiones 2.1 y 2.4).

Siguiente cuaderno: `03_validacion.ipynb`.